In [10]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '7'

In [11]:
import torch
from torchmetrics import StructuralSimilarityIndexMeasure, MeanSquaredError
from torchmetrics.multimodal.clip_score import CLIPScore
from torchmetrics.image.fid import FrechetInceptionDistance
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset

from PIL import Image
from torch.nn import functional as F
from tqdm import tqdm
import json
from utils import image_grid

In [12]:
# load metrics
ssim = StructuralSimilarityIndexMeasure(data_range=1.0)
mse = MeanSquaredError()
clip_score = CLIPScore(model_name_or_path="openai/clip-vit-base-patch16").to('cuda')

In [13]:
img_dir = '相似的几个2'
res_dir = 'results/imgs_eval_only_control_recaption'
names = [name[:-len('.png')] for name in os.listdir(img_dir) if name.endswith('.png')]
result_dict = {name: {'path':os.path.join(img_dir, name+'.png'), 'result_list':[]} for name in names}
for res in os.listdir(res_dir):
    if 'control' in res:
        continue
    for name in result_dict.keys():
        if name in res:
            result_dict[name]['result_list'].append(os.path.join(res_dir, res))
            break
result_dict

{'Snipaste_2023-02-21_13-32-42': {'path': '相似的几个2/Snipaste_2023-02-21_13-32-42.png',
  'result_list': ['results/imgs_eval_only_control_recaption/Snipaste_2023-02-21_13-32-42_stage1_111.png',
   'results/imgs_eval_only_control_recaption/Snipaste_2023-02-21_13-32-42_stage1_222.png',
   'results/imgs_eval_only_control_recaption/Snipaste_2023-02-21_13-32-42_stage1_333.png',
   'results/imgs_eval_only_control_recaption/Snipaste_2023-02-21_13-32-42_stage1_444.png']},
 'Snipaste_2023-02-21_13-33-06': {'path': '相似的几个2/Snipaste_2023-02-21_13-33-06.png',
  'result_list': ['results/imgs_eval_only_control_recaption/Snipaste_2023-02-21_13-33-06_stage1_111.png',
   'results/imgs_eval_only_control_recaption/Snipaste_2023-02-21_13-33-06_stage1_222.png',
   'results/imgs_eval_only_control_recaption/Snipaste_2023-02-21_13-33-06_stage1_333.png',
   'results/imgs_eval_only_control_recaption/Snipaste_2023-02-21_13-33-06_stage1_444.png']},
 'Snipaste_2023-02-21_13-33-51': {'path': '相似的几个2/Snipaste_2023-02-2

In [14]:
sample = result_dict['Snipaste_2023-02-21_13-32-42']
ori_img = Image.open(sample['path']).convert('RGB')
tar_img = Image.open(sample['result_list'][0]).convert('RGB')
tar_img = tar_img.resize(ori_img.size)
#image_grid([ori_img, tar_img], 1, 2)

In [15]:
ssim(transforms.ToTensor()(ori_img).unsqueeze(0), transforms.ToTensor()(tar_img).unsqueeze(0)).item()

0.15885122120380402

In [16]:
mse_res = mse(transforms.ToTensor()(ori_img).unsqueeze(0), transforms.ToTensor()(tar_img).unsqueeze(0)).item()
(1-mse_res)*100

85.89882254600525

In [17]:
# fid for batch
# fid = FrechetInceptionDistance(feature=64, normalize=True)
# fid.update(transforms.ToTensor()(ori_img).unsqueeze(0), real=True)
# fid.update(transforms.ToTensor()(tar_img).unsqueeze(0), real=False)
# fid.compute().item()

In [18]:
clip_score(transforms.ToTensor()(ori_img).unsqueeze(0)*255, transforms.ToTensor()(tar_img).unsqueeze(0)*255).item()

80.40782928466797